In [ ]:
# =======================================================
# Celda 1 Importación de Librerías y Configuración de Path
# =======================================================
import sys
import os
import yaml

try:
    # Añadir el directorio raíz al path para importar módulos de src
    sys.path.append(os.path.abspath(os.path.join('..')))
    from src.A01_data_loader import DataLoader
    print(f"✅ ÉXITO: Entorno configurado. Directorio raíz: {os.path.abspath('..')}")
except Exception as e:
    print(f"❌ ERROR: Error al configurar el entorno: {e}")

In [ ]:
# =======================================================
# Celda 2 Carga de Archivo config.yaml
# =======================================================
try:
    with open('../config/config.yaml', 'r') as f:
        config = yaml.safe_load(f)
    print(f"✅ ÉXITO: Configuración del proyecto '{config['project_name']}' cargada.")
except Exception as e:
    print(f"❌ ERROR: Fallo al leer config.yaml. Verifique la existencia del archivo. Detalle: {e}")

In [ ]:
# =======================================================
# Celda Importación de Archivos CSV y Creación de DataFrames
# =======================================================
try:
    loader = DataLoader(config)
    print("Iniciando creación de objetos en memoria...")

    # Pasamos el nombre que tendrá la variable para que el JSON lo registre
    df_ventas = loader.cargar_csv('raw_sales', 'Ventas Diarias', 'df_ventas')
    print("   -> 'df_ventas' creado y registrado.")

    df_precios = loader.cargar_csv('raw_prices', 'Precios y Costos', 'df_precios')
    print("   -> 'df_precios' creado y registrado.")

    df_cat = loader.cargar_csv('raw_categories', 'Categorías', 'df_cat')
    print("   -> 'df_cat' creado y registrado.")

    print("\n✅ ÉXITO: Los tres DataFrames están listos para la siguiente fase.")
except Exception as e:
    print(f"❌ ERROR CRÍTICO: Fallo en la creación de DataFrames. Detalle: {e}")

In [ ]:
# =======================================================
# Celda 4 Verificación de Objetos Cargados
# =======================================================
try:
    print("Resumen de tablas en memoria:")
    print(f"- df_ventas:  {df_ventas.shape[0]} filas x {df_ventas.shape[1]} columnas")
    print(f"- df_precios: {df_precios.shape[0]} filas x {df_precios.shape[1]} columnas")
    print(f"- df_cat:     {df_cat.shape[0]} filas x {df_cat.shape[1]} columnas")
    print("\n✅ ÉXITO: Los objetos tienen las dimensiones esperadas.")
except NameError as e:
    print(f"❌ ERROR: Uno o más DataFrames no fueron creados correctamente: {e}")

In [ ]:
# =======================================================
# Celda 5 Generación de Archivo JSON de Estado
# =======================================================
try:
    ruta_json = loader.generar_reporte_json()
    print(f"✅ ÉXITO: El reporte de ejecución ha sido guardado.")
    print(f"📄 Ruta del archivo: {ruta_json}")
    
    # Mostrar el contenido del JSON para confirmación inmediata
    with open(ruta_json, 'r') as j:
        import json
        print("\nResumen del JSON generado:")
        print(json.dumps(json.load(j), indent=2))
        
except Exception as e:
    print(f"❌ ERROR: No se pudo consolidar el reporte JSON: {e}")

In [ ]:
# =======================================================
# Celda Importación de StructureValidator
# =======================================================
try:
    from src.A05_data_structure import StructureValidator
    print("✅ ÉXITO: Clase StructureValidator cargada correctamente.")
except Exception as e:
    print(f"❌ ERROR: No se pudo importar el módulo de estructura: {e}")

In [ ]:
# =======================================================
# Celda Validación de Columnas por Tabla
# =======================================================
try:
    validator = StructureValidator(config)
    
    # Definición de columnas esperadas (basado en el análisis previo)
    cols_ventas = ['fecha', 'producto_id', 'categoria', 'tipo_demanda', 'ventas_diarias', 'promocion_activa', 'campana_marketing']
    cols_precios = ['producto_id', 'codigo_categoria', 'nombre_categoria', 'fecha', 'año', 'mes', 'precio_venta_unitario', 'costo_unitario', 'margen_unitario', 'margen_porcentaje', 'cambio_costo_este_mes', 'cambio_precio_este_mes', 'tipo_cambio']
    cols_cat = ['codigo_categoria', 'nombre_categoria', 'producto_id']

    print("Validando estructura de tablas...")
    validator.validar_columnas(df_ventas, "df_ventas", cols_ventas)
    validator.validar_columnas(df_precios, "df_precios", cols_precios)
    validator.validar_columnas(df_cat, "df_cat", cols_cat)
    
    print("✅ ÉXITO: Todas las tablas cumplen con la estructura de columnas esperada.")
except Exception as e:
    print(f"❌ ERROR DE ESTRUCTURA: {e}")

In [ ]:
# =======================================================
# Celda Generación de Reporte a05_status_structure.json
# =======================================================
try:
    ruta_json_struct = validator.generar_reporte_json()
    print(f"✅ ÉXITO: Reporte de estructura generado en: {ruta_json_struct}")
except Exception as e:
    print(f"❌ ERROR: No se pudo guardar el JSON de estructura: {e}")

In [ ]:
# =======================================================
# Celda Importación de QualityValidator (A10)
# =======================================================
try:
    from src.A10_data_quality import QualityValidator
    print("✅ ÉXITO: Clase QualityValidator cargada correctamente.")
except Exception as e:
    print(f"❌ ERROR: No se pudo cargar el módulo A10. Detalle: {e}")

In [ ]:
# =======================================================
# Celda Auditoría Integral de Calidad
# =======================================================
try:
    validador_calidad = QualityValidator(config)
    print("Iniciando escaneo detallado de calidad...")

    validador_calidad.ejecutar_auditoria(df_ventas, "df_ventas")
    validador_calidad.ejecutar_auditoria(df_precios, "df_precios")
    validador_calidad.ejecutar_auditoria(df_cat, "df_cat")

    print("✅ ÉXITO: Auditoría finalizada. Los resultados han sido mapeados.")
except Exception as e:
    print(f"❌ ERROR: Error durante el escaneo: {e}")

In [ ]:
# =======================================================
# Celda Generación de Reporte a10_status_quality.json
# =======================================================
try:
    path_json = validador_calidad.generar_reporte_json()
    print(f"✅ ÉXITO: Reporte de calidad generado en: {path_json}\n")
    
    with open(path_json, 'r') as f:
        log = json.load(f)
        for tabla, data in log['resultados'].items():
            print(f"--- Análisis de {tabla} ---")
            print(f"  * Duplicados: {data['duplicados_en_tabla']}")
            # Mostrar solo una muestra de columnas para no saturar la pantalla
            col_ejemplo = list(data['detalle_por_columna'].keys())[0]
            info_col = data['detalle_por_columna'][col_ejemplo]
            print(f"  * Muestra Columna '{col_ejemplo}':")
            print(f"    - Nulos: {info_col['nulos']}")
            print(f"    - Centinelas: {info_col['centinelas']}")
            print(f"    - Outliers: {info_col['outliers_iqr']}\n")
            
except Exception as e:
    print(f"❌ ERROR: No se pudo generar o leer el reporte: {e}")

In [ ]:
# =======================================================
# Celda Importación de ConsistencyValidator (A15)
# =======================================================
try:
    from src.A15_data_consistency import ConsistencyValidator
    print("✅ ÉXITO: Módulo de Consistencia cargado.")
except Exception as e:
    print(f"❌ ERROR: No se pudo cargar A15: {e}")

In [ ]:
# =======================================================
# Celda Validación de Integridad y Continuidad Temporal
# =======================================================
try:
    validador_cons = ConsistencyValidator(config)
    print("Iniciando validación de relaciones entre tablas...")

    # 1. ¿Ventas vs Categorías?
    validador_cons.validar_integridad_referencial(df_ventas, df_cat, 'producto_id', 'Ventas_vs_Categorias')
    
    # 2. ¿Continuidad de fechas en Ventas?
    validador_cons.validar_continuidad_temporal(df_ventas, "df_ventas")

    print("✅ ÉXITO: Pruebas de consistencia finalizadas.")
except Exception as e:
    print(f"❌ ERROR EN CONSISTENCIA: {e}")

In [ ]:
# =======================================================
# Celda Generación de Reporte a15_status_consistency.json
# =======================================================
try:
    ruta_a15 = validador_cons.generar_reporte_json()
    print(f"✅ ÉXITO: Reporte de consistencia generado en: {ruta_a15}")
    
    # Verificación rápida
    with open(ruta_a15, 'r') as f:
        log_c = json.load(f)
        print("\nResumen Crítico:")
        for k, v in log_c['resultados'].items():
            print(f"- {k}: {v}")
except Exception as e:
    print(f"❌ ERROR: No se pudo guardar el reporte A15: {e}")

In [ ]:
# =======================================================
# Celda Importación de BusinessValidator (A20)
# =======================================================
try:
    from src.A20_data_business import BusinessValidator
    print("✅ ÉXITO: Módulo de Reglas de Negocio cargado correctamente.")
except Exception as e:
    print(f"❌ ERROR: No se pudo cargar A20: {e}")  

In [ ]:
# =======================================================
# Celda Validación de Reglas Comerciales (Desde Config)
# =======================================================
try:
    biz_val = BusinessValidator(config)
    print(f"Iniciando auditoría de negocio (Límite configurado: {config['business_rules']['inactivity_limit_days']} días)...")

    # Ejecutamos las validaciones
    biz_val.validar_margenes(df_precios, "df_precios")
    biz_val.validar_inactividad(df_ventas, "df_ventas") # Ya no pasamos el número aquí

    print("✅ ÉXITO: Auditoría finalizada utilizando parámetros del archivo YAML.")
except Exception as e:
    print(f"❌ ERROR EN REGLAS DE NEGOCIO: {e}")

In [ ]:
# =======================================================
# Celda Generación de Reporte a20_status_business.json
# =======================================================
try:
    ruta_a20 = biz_val.generar_reporte_json()
    print(f"✅ ÉXITO: Reporte de negocio generado en: {ruta_a20}")
    
    # Lectura del JSON para el cierre de fase
    with open(ruta_a20, 'r') as f:
        res_biz = json.load(f)
        print("\nResumen de Salud Comercial:")
        for k, v in res_biz['resultados'].items():
            print(f"- {k}: {v['estado'] if 'estado' in v else v['productos_inactivos_detectados']}")
            
except Exception as e:
    print(f"❌ ERROR: No se pudo consolidar el reporte A20: {e}")